In [ ]:
import os
import subprocess
import sys
import tempfile
from functools import partial

sys.path.insert(0, "/workspace/src")

import fsspec
import matplotlib.pyplot as plt
import numpy as np
import torch

from datasets.cwb import get_zarr_dataset  # noqa: F401
from inference.visualization import (
    plot_comparison,
    plot_ensemble,
)

from physicsnemo import Module
from physicsnemo.diffusion.generate import diffusion_step, regression_step
from physicsnemo.diffusion.samplers import stochastic_sampler

plt.rcParams["animation.embed_limit"] = 100  # MB

# Data: same staging logic as check_cwb_regression.ipynb.
GCS_DATA = "gs://norcorrdiff-us/taiwan_dataset/cwa_dataset/cwa_dataset_cut.zarr"
DATA_PATH = "/workspace/data/cwa_dataset_cut.zarr"  # persistent mount (~/ml-ds_data)

# Diffusion results and the regression checkpoint it was conditioned on.
DIFF_CKPT_DIR = "gs://norcorrdiff-us/results/diffusion_cwb_20260626_095313/checkpoints_diffusion"
REG_CKPT_DIR  = "gs://norcorrdiff-us/results/regression_cwb_20260623_130422/checkpoints_regression"

# Must match the training config (config_training_taiwan_diffusion_gcp.yaml).
HR_MEAN_CONDITIONING = True
# Paper uses 32 members for reported metrics; 4 is fine for quick visual checks.
N_MEMBERS = 4

if not os.path.exists(DATA_PATH):
    print(f"Staging {GCS_DATA}\n     -> {DATA_PATH} (one-time, ~30 GB)...")
    subprocess.run(
        [sys.executable, "/workspace/scripts/stage_zarr.py",
         "--src", GCS_DATA, "--dst", DATA_PATH],
        check=True,
    )

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

DATASET_KWARGS = dict(
    data_path=DATA_PATH,
    in_channels=[0, 1, 2, 3, 4, 9, 10, 11, 12, 17, 18, 19],
    out_channels=[0, 1, 2, 3],
    img_shape_x=128,
    img_shape_y=128,
    add_grid=False,
    ds_factor=1,
    train=False,
    all_times=False,   # 2021 validation split
)

dataset = get_zarr_dataset(**DATASET_KWARGS)

assert len(dataset) > 0, (
    "Validation split is empty — the cut store may not span year 2021."
)

print(f"Dataset length : {len(dataset)}")
print(f"Image shape    : {dataset.image_shape()}")
print(f"Time range     : {dataset.time()[0]}  ->  {dataset.time()[-1]}")

In [ ]:
# Input (ERA5) and output (CWB) channels
input_channels  = dataset.input_channels()
output_channels = dataset.output_channels()

print(f"Input channels  ({len(input_channels)}):")
for ch in input_channels:
    print(f"  {ch.name:<40} level={ch.level}")

print(f"\nOutput channels ({len(output_channels)}):")
for ch in output_channels:
    print(f"  {ch.name:<40} level={ch.level}")

In [ ]:
# Load regression and diffusion checkpoints from GCS.
# Checkpoints are named <ModelClass>.<...>.<nimg>.mdlus; pick the highest nimg.
fs = fsspec.filesystem("gs")

def _load_latest_ckpt(ckpt_dir: str) -> Module:
    """Download the highest-nimg .mdlus checkpoint to a temp file and load it."""
    ckpts = [p for p in fs.ls(ckpt_dir) if p.endswith(".mdlus")]
    assert ckpts, f"No .mdlus checkpoints found in {ckpt_dir}"
    latest = max(ckpts, key=lambda p: int(p.split(".")[-2]))
    print(f"Loading: gs://{latest}")
    with tempfile.NamedTemporaryFile(suffix=".mdlus", delete=False) as tmp:
        tmp_path = tmp.name
    try:
        with fs.open(latest, "rb") as remote_f, open(tmp_path, "wb") as local_f:
            local_f.write(remote_f.read())
        return Module.from_checkpoint(tmp_path)
    finally:
        os.unlink(tmp_path)


def _prep_net(net: Module) -> Module:
    net = net.eval().to(device).to(memory_format=torch.channels_last)
    if hasattr(net, "amp_mode"):
        net.amp_mode = False
    return net


net_reg = _prep_net(_load_latest_ckpt(REG_CKPT_DIR))
net_res = _prep_net(_load_latest_ckpt(DIFF_CKPT_DIR))
print("Both networks loaded")

# Stochastic SDE sampler — matches the paper (18 steps) and config_generate_taiwan.yaml.
# The deterministic ODE path (9-step Euler) collapses ensemble spread, hurting CRPS.
sampler_fn = partial(stochastic_sampler, num_steps=18, patching=None)

In [ ]:
C_out = len(dataset.output_channels())
H, W = dataset.image_shape()


@torch.no_grad()
def predict_diffusion(idx, n_members=N_MEMBERS):
    """Return (truth, ensemble), truth is (C, H, W), ensemble is (n_members, C, H, W) — physical units."""
    target, input_ = dataset[idx]
    img_lr = input_[None].to(device).float().to(memory_format=torch.channels_last)

    # Regression mean: used both as conditioning and as the base for ensemble members.
    image_reg = regression_step(
        net=net_reg,
        img_lr=img_lr,
        latents_shape=(n_members, C_out, H, W),
        lead_time_label=None,
    )  # (n_members, C_out, H, W)

    mean_hr = image_reg[0:1] if HR_MEAN_CONDITIONING else None

    image_res = diffusion_step(
        net=net_res,
        sampler_fn=sampler_fn,
        img_shape=(H, W),
        img_out_channels=C_out,
        rank_batches=[torch.arange(n_members)],
        img_lr=img_lr.expand(n_members, -1, -1, -1).to(memory_format=torch.channels_last),
        rank=0,
        device=device,
        mean_hr=mean_hr,
        lead_time_label=None,
    )  # (n_members, C_out, H, W)

    ensemble = image_reg + image_res  # regression mean + stochastic residuals

    truth = dataset.denormalize_output(target[None].numpy())[0]
    ensemble_phys = dataset.denormalize_output(ensemble.cpu().numpy())
    return truth, ensemble_phys


# Sanity check on the first validation sample.
_truth, _ens = predict_diffusion(0)
print(f"truth shape    : {_truth.shape}")
print(f"ensemble shape : {_ens.shape}  (n_members, C_out, H, W)")
print(f"finite values  : truth={np.isfinite(_truth).all()}, ensemble={np.isfinite(_ens).all()}")

In [ ]:
import datetime as dt


def idx_for_time(time_str: str) -> int:
    """Return the dataset index for an ISO 8601 timestamp string.

    dataset.time() returns cftime objects; compare by field to avoid calendar issues.
    Raises ValueError if the timestamp isn't in the validation split.
    """
    target = dt.datetime.strptime(time_str, "%Y-%m-%dT%H:%M:%S")
    for i, t in enumerate(dataset.time()):
        if (t.year, t.month, t.day, t.hour) == (target.year, target.month, target.day, target.hour):
            return i
    raise ValueError(f"{time_str} not found in the validation dataset. "
                     f"Time range: {dataset.time()[0]} -> {dataset.time()[-1]}")


# Verify: first validation sample and the timestamp from config_generate_taiwan.yaml
print(f"dataset.time()[0]       : {dataset.time()[0]}")
print(f"dataset.time()[-1]      : {dataset.time()[-1]}")
print(f"idx for 2021-02-02T00   : {idx_for_time('2021-02-02T00:00:00')}")


In [ ]:
# Use the same timestamp as config_generate_taiwan.yaml for a direct apples-to-apples comparison.
# idx=0 is the FIRST 2021 sample, which may not be 2021-02-02 — use idx_for_time() instead.
COMPARE_TIME = "2021-09-12T10:00:00"
idx = idx_for_time(COMPARE_TIME)
truth, ensemble = predict_diffusion(idx, n_members=64)

In [ ]:
# ERA5 input panel: find the input channel whose name matches output channel channel_idx=1.
_out_ch = dataset.output_channels()[1]
_in_names = [ch.name for ch in dataset.input_channels()]
_era5_label = f"{_out_ch.name} {_out_ch.level}".strip()
if _out_ch.name in _in_names:
    _era5_idx = _in_names.index(_out_ch.name)
    _, _input_raw = dataset[idx]
    _input_phys = dataset.denormalize_input(_input_raw[None].numpy())  # (1, C_in, H_in, W_in)
    _input_lr = _input_phys[0, _era5_idx]
else:
    print(f"Warning: output channel '{_out_ch.name}' not found in ERA5 input channels; skipping ERA5 panel.")
    _input_lr = None

In [ ]:
plot_ensemble(
    truth, ensemble, dataset.output_channels(), dataset.time()[idx],
    channel_idx=1,
    input_lr=_input_lr,
    input_label=f"ERA5 input\n{_era5_label}",
)

In [ ]:
# Regression mean vs truth for comparison (channel_idx=1 = temperature_2m).
plot_comparison(
    truth, ensemble.mean(axis=0), dataset.output_channels(), dataset.time()[idx],
    channel_idx=1,
    input_lr=_input_lr,
    input_label=f"ERA5 input\n{_era5_label}",
)